# Week 6 — Fari Churn & Retention Analytics (Strict Requirements Revision)
All contract revenue and interim contract events are **synthetic assumptions**. The Week 2 source is preserved. This notebook computes results and Plotly charts directly.

In [1]:
from pathlib import Path
import pandas as pd, json
import plotly.express as px
from src.churn_model import (build_asof_features, construct_matured_labels, leave_one_deployment_out, evaluate_predictions, train_full_and_score, calculate_revenue_risk, select_top_k, model_driver_coefficients, lifecycle_as_of, at_risk_quartile_audit, risk_concentration)
from src.contract_events import synthetic_contract_retention
P=Path.cwd(); O=P/"outputs"
obs=pd.read_csv(P/"data/W02_Robot_Deployment_Dataset.csv")
contracts=pd.read_csv(P/"data/W06_Fari_Contract_Metadata_SYNTHETIC_ASSUMPTIONS.csv")
events=pd.read_csv(P/"data/W06_Synthetic_Contract_Events_SCENARIO.csv")

## 1. Contract retention: separate hypothetical event scenario
These are modeled **contract-active** proportions using explicit hypothetical contract exits, NOT observed renewal rates. Original records only contain final renewal labels. Day- and month-based horizons are distinct.

In [2]:
ret=synthetic_contract_retention(events)
ret["sortkey"]=ret.horizon.map({"30d":0,"60d":1,"90d":2,"3m":3,"6m":4,"12m":5})
ret=ret.sort_values("sortkey")
ret.drop(columns="sortkey").to_csv(O/"W06_SYNTHETIC_Contract_Retention_30_60_90_3_6_12.csv",index=False)
display(ret)

,assumed_start_cohort,horizon,eligible,retained,retention_rate,metric_type,sortkey
1,2025Q1,30d,10,10,1.0,SIMULATED_CONTRACT_RETENTION_NOT_OBSERVED,0
3,2025Q1,60d,10,10,1.0,SIMULATED_CONTRACT_RETENTION_NOT_OBSERVED,1
5,2025Q1,90d,10,10,1.0,SIMULATED_CONTRACT_RETENTION_NOT_OBSERVED,2
2,2025Q1,3m,10,10,1.0,SIMULATED_CONTRACT_RETENTION_NOT_OBSERVED,3
4,2025Q1,6m,10,9,0.9,SIMULATED_CONTRACT_RETENTION_NOT_OBSERVED,4
0,2025Q1,12m,10,8,0.8,SIMULATED_CONTRACT_RETENTION_NOT_OBSERVED,5


In [3]:
for horizons,filename in [(["30d","60d","90d"],"W06_Simulated_30_60_90_Day_Retention.html"),(["3m","6m","12m"],"W06_Simulated_3_6_12_Month_Retention.html")]:
    d=ret[ret.horizon.isin(horizons)].copy()
    d["horizon"]=pd.Categorical(d.horizon,categories=horizons,ordered=True)
    fig=px.line(d.sort_values("horizon"),x="horizon",y="retention_rate",color="assumed_start_cohort",markers=True,range_y=[0,1.05],title="SYNTHETIC scenario retention (not observed contract renewals)")
    fig.write_html(O/filename,include_plotlyjs="cdn")
    fig.show()

## 2. Leakage-safe retrospective churn model
Features use observations on or before September 1, 2025; final December renewal labels are only used after maturity. Deployment-level LOOCV is exploratory on 10 synthetic deployments.

In [4]:
features=build_asof_features(obs,contracts,"2025-09-01")
labels=construct_matured_labels(obs,contracts,outcome_as_of="2025-12-31")
oof=leave_one_deployment_out(features,labels)
metrics=evaluate_predictions(oof,k=10)
print(metrics)
scores=train_full_and_score(features,labels)
display(scores)

{'n_deployments': 10, 'n_churned': 2, 'auc': 0.8125, 'precision_at_0_5': 0.5, 'recall_at_0_5': 0.5, 'top_k_precision': 0.2, 'effective_k': 10, 'threshold': 0.5}


,deployment_id,prediction_as_of,churn_probability_exploratory,score_type
0,FARI-008,2025-09-01,0.887119,IN_SAMPLE_EXPLORATORY_NOT_VALIDATION
1,FARI-003,2025-09-01,0.793901,IN_SAMPLE_EXPLORATORY_NOT_VALIDATION
2,FARI-001,2025-09-01,0.454776,IN_SAMPLE_EXPLORATORY_NOT_VALIDATION
3,FARI-004,2025-09-01,0.268349,IN_SAMPLE_EXPLORATORY_NOT_VALIDATION
4,FARI-006,2025-09-01,0.167783,IN_SAMPLE_EXPLORATORY_NOT_VALIDATION
5,FARI-005,2025-09-01,0.130434,IN_SAMPLE_EXPLORATORY_NOT_VALIDATION
6,FARI-002,2025-09-01,0.109681,IN_SAMPLE_EXPLORATORY_NOT_VALIDATION
7,FARI-007,2025-09-01,0.075337,IN_SAMPLE_EXPLORATORY_NOT_VALIDATION
8,FARI-009,2025-09-01,0.052773,IN_SAMPLE_EXPLORATORY_NOT_VALIDATION
9,FARI-010,2025-09-01,0.016628,IN_SAMPLE_EXPLORATORY_NOT_VALIDATION


## 3. Simulated remaining LTV, revenue at risk, and Top-K

In [5]:
risk=calculate_revenue_risk(scores,contracts,"2025-09-01")
top=select_top_k(risk,k=10)
display(top[["intervention_rank","deployment_id","expected_revenue_at_risk_usd","residual_ltv_usd"]])
fig=px.bar(top,x="deployment_id",y="expected_revenue_at_risk_usd",title="SIMULATED Top-K potential revenue loss (in-sample model scores)")
fig.show()

,intervention_rank,deployment_id,expected_revenue_at_risk_usd,residual_ltv_usd
0,1,FARI-008,8516.338662,1083.661338
1,2,FARI-003,6986.331811,1813.668189
2,3,FARI-001,2910.568684,3489.431316
3,4,FARI-004,1610.096482,4389.903518
4,5,FARI-006,1140.923223,5659.076777
5,6,FARI-005,1043.471611,6956.528389
6,7,FARI-002,789.701172,6410.298828
7,8,FARI-007,572.563894,7027.436106
8,9,FARI-009,443.289004,7956.710996
9,10,FARI-010,116.395024,6883.604976


## 4. Driver associations and concentration checks
Regression coefficients are associations, not causal proof. The required 25% concentration threshold may not hold; report its actual value without changing source data.

In [6]:
driver=model_driver_coefficients(features,labels)
fig=px.bar(driver,x="standardized_coefficient",y="feature",orientation="h",title="Standardized churn model associations (not causal drivers)")
fig.show()
lifecycle=lifecycle_as_of(obs,contracts,"2025-09-01")
audit=at_risk_quartile_audit(risk,lifecycle.loc[lifecycle.lifecycle_stage.eq("At-Risk"),"deployment_id"])
concentration=risk_concentration(risk)
print("At-risk top-LTV-quartile 25% condition:",audit)
print("Early high-value risk:",concentration["early_high_value_risk_share"],"vs prior 30% benchmark")

At-risk top-LTV-quartile 25% condition: {'at_risk_pool_size': 3, 'top_quartile_size': 1, 'share': 0.0836760145609218, 'meets_25pct': False, 'reason': 'Evaluated on at-risk pool; numerator and denominator aligned'}
Early high-value risk: 0.0 vs prior 30% benchmark


## Limitations
- Hypothetical intermediate churn dates were added only to a **separate synthetic scenario**, not observed from Fari contracts.
- The 25% at-risk concentration requirement **does not pass** in the original scenario, and the early-risk share is 0% under the original assumed dates.
- Only one assumed original cohort exists; no authentic multi-cohort comparison can be made.
- Ten deployments and two non-renewals cannot establish external performance or causal explanations.